In [1]:
import warnings
warnings.filterwarnings('ignore')
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](../gen_demo_factor_data.py) 脚本生成示例数据。

# HDF5DB

![HDF5 因子库](../images/HDF5因子库.png)

In [2]:
from QuantStudio.Factor.HDF5DB import HDF5DB

HDB = HDF5DB(args={"MainDir": "../Data/HDF5"}).connect()
print(qs_help(HDB))

类型: HDF5DB
模块: QuantStudio.Factor.HDF5DB
QS 对象类型: 因子库
QS 对象名称: HDF5DB
QSID: 554967d75d893bd0098970919e68d4a166e9550ec9bd4574af069dcbbef513c8
参数集:
    * Name(名称): <class 'str'>, 默认值 HDF5DB, 当前取值: 'HDF5DB'
    * MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录, 当前取值: ..\Data\HDF5
    * LockDir(锁目录): typing.Optional[typing.Annotated[pathlib.Path, PathType(path_type='dir')]], 默认值 None, 存放锁文件的目录, 默认 None 表示和主目录相同, 当前取值: None
    * FileOpenRetryNum(文件打开重试次数): typing.Union[int, float], 默认值 inf, 打开数据文件错误时的重试次数, 当前取值: inf
    * ProcessLock(进程锁): <class 'bool'>, 默认值 True, 是否添加进程锁用于防止多进程间读写冲突, 当前取值: True
说明文档:
    基于 HDF5 文件的因子库
    主目录下的每个文件夹表示一张因子表, 每个文件夹下扩展名为 hdf5 的 [HDF5 文件](https://www.hdfgroup.org/) 存储了一个因子的数据。
    每个 HDF5 因子文件有三个 Dataset:
        * ID: 存储因子的 ID 序列数据, shape=(None,), dtype=String, 编码为 utf-8。
        * DateTime: 存储因子的时点序列数据, shape=(None,), dtype=float, 时点转换成 timestamp 存储。
        * Data: 存储因子数据, shape=(None, None), 行数等于 DateTime 的长度, 列数等于 ID 的长度。double 类型的因子数据存储为 float64 

## 因子表

In [3]:
# 因子表列表
print("因子表 : ", HDB.TableNames)

因子表 :  ['index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']


In [4]:
FT = HDB.getTable("stock_cn_day_bar", args={"LookBack": 4})
print(qs_help(FT))

类型: HDF5FactorTable
模块: QuantStudio.Factor.HDF5DB
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_cn_day_bar
QSID: d9ea274f8efdde70468367aef01213434f85bed1f5e25fb69ac2fc61af318ea5
参数集:
    * Name(名称): <class 'str'>, 默认值 Node, 当前取值: 'stock_cn_day_bar'
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 当前取值: 4
    * OnlyStartLookBack(只起始日回溯): <class 'bool'>, 默认值 False, 如果为 True, 表示只对提取数据的第一个时点进行缺失填充, 之后的时点不填充, 当前取值: False
    * OnlyLookBackNontarget(只回溯非目标日): <class 'bool'>, 默认值 False, 如果为 True, 表示只用不在提取时点序列中的数据进行缺失填充, 当前取值: False
    * OnlyLookBackDT(只回溯时点): <class 'bool'>, 默认值 False, 如果为 True, 表示所有 ID 统一沿着时点字段进行回溯填充, 不单独填充, 当前取值: False
    * TargetDT(目标时点): typing.Optional[datetime.datetime], 默认值 None, 非 None 表示只取该时点的值返回, 当前取值: None
说明文档:
    HDF5DB 库中因子表


In [5]:
# 因子列表
print(f"因子表 '{FT.Name}' 中的因子 : ", FT.FactorNames)

因子表 'stock_cn_day_bar' 中的因子 :  ['amount', 'close', 'high', 'low', 'open', 'volume']


In [6]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

print("-" * 10)
print("固定 ID 的切片数据: ", Data.iloc[:, :, 0], sep="\n")

因子表数据
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 1 (minor_axis)
Items axis: open to close
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000001.SZ
----------
固定 ID 的切片数据: 
                open     close
2025-01-01  5.488135  8.115185
2025-01-02  9.786183  0.352198
2025-01-03  3.595079  6.019437
2025-01-04  1.589696  3.936297
2025-01-05  3.179832  4.884425
